In [2]:
import requests
import datetime
from tqdm import tqdm

In [ ]:
# Replace with your actual API token from Travelpayouts 
# You can just google "aviaseils API", easy to register and all is free, 
# but this data is some kind of cache, so it is not too representative, but still good enough 
TOKEN = "ergergerger"

# Construct the URL with the desired parameters:
# - origin: Moscow (MOW)
# - destination: Tokyo (TYO)
# - departure date: 2025-03-28
# - return date: 2025-04-10
# - one_way=false for round-trip offers
# - sorting: Sorting by price, 
# - direct=false: including non-direct flights, 
# - cy: currency in USD (RUB)
# - limit, page: how many tickets will be loaded from pages (list sorted by sorting param) 
url = (
    "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
    "origin=MOW&destination=TYO&departure_at=2026-07-26&return_at=2026-08-05"
    "&unique=false&sorting=price&direct=false&cy=RUB&limit=30&page=1&one_way=false"
    f"&token={TOKEN}"
)


Test check of API.

In [41]:
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    if data.get("success") and data.get("data"):
        flights = data["data"]
        print("\n✈️  Cheapest Flights from Moscow (MOW) to Tokyo (TYO):\n")
        for idx, flight in enumerate(flights, 1):
            price = flight.get("price", "N/A")
            airline = flight.get("airline", "N/A")
            flight_number = flight.get("flight_number", "N/A")
            departure_at = flight.get("departure_at", "N/A")
            return_at = flight.get("return_at", "N/A")
            # Calculate total transfers (for both outbound and return legs)
            transfers = flight.get("transfers", 0)
            return_transfers = flight.get("return_transfers", 0)
            total_transfers = transfers + return_transfers
            link = flight.get("link", "")
            full_link = f"https://www.aviasales.ru{link}" if link else "N/A"

            print(f"🔹 Option {idx}:")
            print(f"   - 💰 Price: ${price}")
            print(f"   - ✈️ Airline: {airline} (Flight No: {flight_number})")
            print(f"   - 🛫 Departure: {departure_at}")
            print(f"   - 🛬 Return: {return_at}")
            print(f"   - ⏳ Total Transfers: {total_transfers}")
            print(f"   - 🔗 Ticket Link: {full_link}\n")
    else:
        print("❌ No flight data found for the specified dates and parameters.")
else:
    print(f"❌ Error {response.status_code}: {response.text}")



✈️  Cheapest Flights from Moscow (MOW) to Tokyo (TYO):

🔹 Option 1:
   - 💰 Price: $72480
   - ✈️ Airline: WY (Flight No: 184)
   - 🛫 Departure: 2026-07-26T22:05:00+03:00
   - 🛬 Return: 2026-08-05T10:20:00+09:00
   - ⏳ Total Transfers: 4
   - 🔗 Ticket Link: https://www.aviasales.ru/search/MOW2607TYO05081?t=MH17851035001785261900002280SVOMCTKULNRT17859252001786050300002445NRTKULMCTSVO_28c01db8a91399eb898e89acd02ae424_72408&search_date=14042026&expected_price_uuid=019d8c06-292e-8030-ae4d-4b56051e702a&static_fare_key=TY%7CP0%7CH1%7CL1_1_0%7CCH0%7CR0%7CTBC1&expected_price_source=share&expected_price_currency=rub&expected_price=72480

🔹 Option 2:
   - 💰 Price: $74674
   - ✈️ Airline: MU (Flight No: 592)
   - 🛫 Departure: 2026-07-26T19:05:00+03:00
   - 🛬 Return: 2026-08-05T16:55:00+09:00
   - ⏳ Total Transfers: 2
   - 🔗 Ticket Link: https://www.aviasales.ru/search/MOW2607TYO05081?t=MU17850927001785241800002125SVOPVGSHAHND17859489001786039800001875NRTPVGSVO_0398714fa6be27ed623a081910cc68e2_74

# Main search of tickets

Both sides tickets (3 cities, 12-17 days that should overlap 2 days from date window (2025, 3, 28)-(2025, 4, 9))

In [35]:
transfters_limit = 5
price_limit = 70000

In [36]:
# Search parameters
origin = "MOW"
destinations = ["TYO", "NGO", "OSA"]  # Tokyo, Nagoya, Osaka (IATA codes)
min_duration = 10
max_duration = 12

# The window that must overlap with the trip (at least 2 days)
window_start = datetime.date(2026, 7, 29)
window_end = datetime.date(2026, 8, 5)
window_end_plus = window_end + datetime.timedelta(days=1)  # for inclusive overlap

# Range of possible departure dates (from 2025-03-15 to 2025-04-07)
start_departure = datetime.date(2026, 7, 27)
end_departure = datetime.date(2026, 8, 15)
days_range = (end_departure - start_departure).days + 1

# Pre-calculate all query combinations: (destination, trip_duration, departure_date)
queries = [
    (dest, duration, start_departure + datetime.timedelta(days=offset))
    for dest in destinations
    for duration in range(min_duration, max_duration + 1)
    for offset in range(days_range)
]

queries[0], len(queries)

(('TYO', 10, datetime.date(2026, 7, 27)), 180)

In [37]:
# queries = [
#     (dest, duration, start_departure + datetime.timedelta(days=offset))
#     for dest in destinations
#     for duration in range(min_duration, max_duration + 1)
#     for offset in [0,3,4,5,6,7]
# ]
# queries[0], len(queries)

In [38]:
results = []  # To store matching flight offers

# Create a session to reuse connections
session = requests.Session()

def build_api_url(origin, destination, departure_date, return_date):
    """Return the API URL for given parameters."""
    d_str = departure_date.strftime("%Y-%m-%d")
    return_str = return_date.strftime("%Y-%m-%d")
    url = (
        "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
        f"origin={origin}&destination={destination}&departure_at={d_str}&return_at={return_str}"
        "&unique=false&sorting=price&direct=false&cy=RUB&limit=5&page=1&one_way=false"
        f"&token={TOKEN}"
    )
    return url

# Iterate over all query combinations using a tqdm progress bar.
for destination, trip_duration, departure_date in tqdm(queries, desc="Processing queries"):
    return_date = departure_date + datetime.timedelta(days=trip_duration)
    # Calculate overlap between the trip period and the window
    overlap_start = max(departure_date, window_start)
    overlap_end = min(return_date, window_end_plus)
    overlap_days = (overlap_end - overlap_start).days
    if overlap_days < 2:
        continue

    url = build_api_url(origin, destination, departure_date, return_date)
    try:
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("success") and data.get("data"):
                for flight in data["data"]:
                    price = flight.get("price", 0)
                    try:
                        ntransfers = int(flight.get("transfers", 0) + flight.get("return_transfers", 0))
                    except Exception:
                        ntransfers = 0
                    if price < price_limit and ntransfers < transfters_limit:
                        flight_info = {
                            "destination": destination,
                            "departure": departure_date.strftime("%Y-%m-%d"),
                            "return": return_date.strftime("%Y-%m-%d"),
                            "price": price,
                            "airline": flight.get("airline", "N/A"),
                            "flight_number": flight.get("flight_number", "N/A"),
                            "departure_at": flight.get("departure_at", "N/A"),
                            "return_at": flight.get("return_at", "N/A"),
                            "transfers": ntransfers,
                            "link": f"https://www.aviasales.ru{flight.get('link', '')}"
                        }
                        results.append(flight_info)
                        # Uncomment the following line to log each matching price:
                        # print(f"Found price: {price}, in {destination}")
        else:
            print(f"Error {response.status_code} for URL: {url}")
    except Exception as e:
        print(f"Request failed: {e}")

# Sort results by price (lowest first)
results.sort(key=lambda x: x["price"])

Processing queries: 100%|██████████| 180/180 [00:23<00:00,  7.79it/s]


In [39]:
# Print the results in a readable format
print(f"\n✈️  Flights found under {price_limit} rub:\n")
if results:
    for idx, flight in enumerate(results, 1):
        print(f"🔹 Option {idx}:")
        print(f"   - Destination: {flight['destination']}")
        print(f"   - Price: {flight['price']} rub")
        print(f"   - Trip Dates: {flight['departure']} to {flight['return']}")
        print(f"   - Airline: {flight['airline']} (Flight No: {flight['flight_number']})")
        print(f"   - Departure Time: {flight['departure_at']}")
        print(f"   - Return Time: {flight['return_at']}")
        print(f"   - Total Transfers: {flight['transfers']}")
        print(f"   - Ticket Link: {flight['link']}\n")
else:
    print("No flights found that meet the criteria.")



✈️  Flights found under 70000 rub:

🔹 Option 1:
   - Destination: OSA
   - Price: 64905 rub
   - Trip Dates: 2026-08-03 to 2026-08-14
   - Airline: EY (Flight No: 842)
   - Departure Time: 2026-08-03T12:40:00+03:00
   - Return Time: 2026-08-14T18:40:00+09:00
   - Total Transfers: 2
   - Ticket Link: https://www.aviasales.ru/search/MOW0308OSA14081?t=EY17857608001785844200001030SVOAUHKIX17867328001786800600001490KIXAUHSVO_6a8176ad277d92aec985c2965f15b8cc_64948&search_date=14042026&expected_price_uuid=019d8c0a-cf4b-8030-a38d-ac6e3745559f&static_fare_key=TY%7CP0%7CH1%7CL0%7CCH0%7CR0%7CTBC0&expected_price_source=share&expected_price_currency=rub&expected_price=64905

🔹 Option 2:
   - Destination: OSA
   - Price: 64905 rub
   - Trip Dates: 2026-08-03 to 2026-08-14
   - Airline: EY (Flight No: 842)
   - Departure Time: 2026-08-03T12:40:00+03:00
   - Return Time: 2026-08-14T18:40:00+09:00
   - Total Transfers: 2
   - Ticket Link: https://www.aviasales.ru/search/MOW0308OSA14081?t=EY17857608001

# Separate tickets 

Two independed tickets but with same pattern (3 cities, 12-17 days that should overlap 2 days from date window (2025, 3, 28)-(2025, 4, 9))

I found for my routs they has poor representation, so its gives worse in price and less results.

In [21]:
# Search criteria:
min_duration = 10
max_duration = 12
price_threshold = price_limit  # Total combined price in USD

# Target window: The trip must overlap this window (28 March–9 April) by at least 2 days.
window_start = datetime.date(2026, 8, 1)
window_end = datetime.date(2026, 8, 3)

# Date range for searches (used for both outbound and inbound flights)
start_date = datetime.date(2026, 7, 27)
end_date = datetime.date(2026, 8, 15)
days_range = (end_date - start_date).days + 1

# Create a session for improved performance
session = requests.Session()

def search_one_way(origin, destination, search_date):
    """
    Search for one-way flights for a given search_date between origin and destination.
    Returns a list of flights that (after conversion) have exactly one transfer.
    """
    flights = []
    date_str = search_date.strftime("%Y-%m-%d")
    url = (
        "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
        f"origin={origin}&destination={destination}&departure_at={date_str}"
        "&unique=false&sorting=price&direct=false&cy=RUB&limit=10&page=1&one_way=true"
        f"&token={TOKEN}"
    )
    try:
        response = session.get(url, timeout=10)
        # Uncomment the following line for debugging a specific date:
        # print(f"DEBUG: URL: {url}\nResponse: {response.text}\n")
        if response.status_code == 200:
            data = response.json()
            if data.get("success") and data.get("data"):
                for flight in data["data"]:
                    try:
                        transfers = int(flight.get("transfers", 0))
                    except Exception:
                        transfers = 0
                    # Only accept flights with exactly 1 transfer.
                    if transfers < transfters_limit :
                        flights.append({
                            "from": origin,
                            "to": destination,
                            "departure_date": search_date,
                            "price": flight.get("price", 0),
                            "airline": flight.get("airline", "N/A"),
                            "flight_number": flight.get("flight_number", "N/A"),
                            "departure_at": flight.get("departure_at", "N/A"),
                            "link": f"https://www.aviasales.ru{flight.get('link', '')}"
                        })
        else:
            print(f"Error {response.status_code} for URL: {url}")
    except Exception as e:
        print(f"Request failed for {url}: {e}")
    return flights

# -------------------------
# Outbound search: MOW -> OSA
# -------------------------
print("Searching outbound flights (MOW -> OSA)...")
outbound_results_osa = []
for day_offset in tqdm(range(days_range), desc="Outbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    outbound_results_osa.extend(search_one_way("MOW", "OSA", current_date))

# -------------------------
# Outbound search: MOW -> TYO
# -------------------------
print("Searching outbound flights (MOW -> TYO)...")
outbound_results_tyo = []
for day_offset in tqdm(range(days_range), desc="Outbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    outbound_results_tyo.extend(search_one_way("MOW", "TYO", current_date))

# -------------------------
# Outbound search: MOW -> NGO
# -------------------------
print("Searching outbound flights (MOW -> NGO)...")
outbound_results_ngo = []
for day_offset in tqdm(range(days_range), desc="Outbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    outbound_results_ngo.extend(search_one_way("MOW", "NGO", current_date))

# -------------------------
# Inbound search: TYO -> MOW
# -------------------------
print("Searching inbound flights (TYO -> MOW)...")
inbound_results_tyo = []
for day_offset in tqdm(range(days_range), desc="Inbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    inbound_results_tyo.extend(search_one_way("TYO", "MOW", current_date))

# -------------------------
# Inbound search: OSA -> MOW
# -------------------------
print("Searching inbound flights (OSA -> MOW)...")
inbound_results_osa = []
for day_offset in tqdm(range(days_range), desc="Inbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    inbound_results_osa.extend(search_one_way("OSA", "MOW", current_date))

# -------------------------
# Inbound search: NGO -> MOW
# -------------------------
print("Searching inbound flights (NGO -> MOW)...")
inbound_results_ngo = []
for day_offset in tqdm(range(days_range), desc="Inbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    inbound_results_ngo.extend(search_one_way("NGO", "MOW", current_date))


Searching outbound flights (MOW -> OSA)...


Outbound search: 100%|██████████| 20/20 [00:05<00:00,  3.63it/s]


Searching outbound flights (MOW -> TYO)...


Outbound search: 100%|██████████| 20/20 [00:05<00:00,  3.64it/s]


Searching outbound flights (MOW -> NGO)...


...

Inbound search: 100%|██████████| 20/20 [00:06<00:00,  3.04it/s]


Searching inbound flights (OSA -> MOW)...


Inbound search: 100%|██████████| 20/20 [00:05<00:00,  3.50it/s]


Searching inbound flights (NGO -> MOW)...


Inbound search: 100%|██████████| 20/20 [00:06<00:00,  3.31it/s]


In [23]:

# -------------------------
# Combine outbound and inbound flights into round-trip pairs
# -------------------------
combined_results = []
print("Combining outbound and inbound flights...")
results = [
    [outbound_results_osa, inbound_results_tyo], 
    [outbound_results_osa, inbound_results_ngo], 
    [outbound_results_ngo, inbound_results_tyo], 
    [outbound_results_ngo, inbound_results_osa], 
    [outbound_results_tyo, inbound_results_osa],
    [outbound_results_tyo, inbound_results_ngo]
    ] 
for outbound_results, inbound_results in results:
    for out_flight in tqdm(outbound_results, desc="Combining flights (outbound)"):
        for in_flight in inbound_results:
            # Ensure the inbound flight departs after the outbound flight.
            trip_duration = (in_flight["departure_date"] - out_flight["departure_date"]).days
            if min_duration <= trip_duration <= max_duration:
                trip_start = out_flight["departure_date"]
                trip_end = in_flight["departure_date"]
                overlap_start = max(trip_start, window_start)
                overlap_end = min(trip_end, window_end)
                overlap_days = (overlap_end - overlap_start).days + 1 if overlap_end >= overlap_start else 0
                if overlap_days >= 2:
                    total_price = out_flight["price"] + in_flight["price"]
                    if total_price < price_threshold:
                        combined_results.append({
                            "outbound": out_flight,
                            "inbound": in_flight,
                            "trip_duration": trip_duration,
                            "total_price": total_price
                        })

# -------------------------
# Print combined results
# -------------------------
print(f"\n✈️  Combined One-Way Ticket Pairs (Total Price under {price_limit} rub):\n")
if combined_results:
    combined_results.sort(key=lambda x: x["total_price"])
    for idx, combo in enumerate(combined_results, 1):
        out_flight = combo["outbound"]
        in_flight = combo["inbound"]
        print(f"🔹 Option {idx}:")
        print(f"   Outbound ({out_flight['from']} -> {out_flight['to']}):")
        print(f"       * Departure Date: {out_flight['departure_date']}")
        print(f"       * Price: ${out_flight['price']}")
        print(f"       * Airline: {out_flight['airline']} (Flight No: {out_flight['flight_number']})")
        print(f"       * Departure Time: {out_flight['departure_at']}")
        print(f"       * Ticket Link: {out_flight['link']}")
        print(f"   Inbound ({in_flight['from']} -> {in_flight['to']}):")
        print(f"       * Departure Date: {in_flight['departure_date']}")
        print(f"       * Price: ${in_flight['price']}")
        print(f"       * Airline: {in_flight['airline']} (Flight No: {in_flight['flight_number']})")
        print(f"       * Departure Time: {in_flight['departure_at']}")
        print(f"       * Ticket Link: {in_flight['link']}")
        print(f"   Trip Duration: {combo['trip_duration']} days")
        print(f"   Total Price: ${combo['total_price']}\n")
else:
    print("No flight combinations found that meet the criteria.")


Combining outbound and inbound flights...


Combining flights (outbound): 100%|██████████| 20/20 [00:00<00:00, 100462.37it/s]


✈️  Combined One-Way Ticket Pairs (Total Price under 70000 rub):

No flight combinations found that meet the criteria.


# Multi-City Scraped Results Parsing
Reads the JSON file generated by `parse_multi_city.py`.

In [60]:
import json

price_limit_rubles = 75000

print(f"\n✈️  Best Multi-City Flights found under {price_limit_rubles} rub:\n")

try:
    with open("multi_city_results.json", "r", encoding="utf-8") as f:
        multi_city_results = json.load(f)
        
    # Filter out results where min_price is None or larger than limit
    valid_results = [r for r in multi_city_results if r.get("min_price") and r["min_price"] <= price_limit_rubles and r.get('transfers', 'Unknown') < 3]
    valid_results.sort(key=lambda x: x["min_price"])
    
    if valid_results:
        for idx, flight in enumerate(valid_results, 1):
            print(f"Option {idx}:")
            print(f"   -  flight: {flight['origin']}-{flight['dest1']} + {flight['dest2']}-{flight['origin']}")
            print(f"   - 💰 Price: {flight['min_price']} rub")
            
            transfers = flight.get('transfers', 'Unknown')
            print(f"   - ⏳ Total Transfers (both ways): {transfers}")
            
            print(f"   - 🛫 Departure Date: {flight['departure']}")
            print(f"   - 🛬 Return Date: {flight['return']}")
            
            link = flight.get('ticket_link', flight.get('url'))
            print(f"   - 🔗 Ticket Link: {link}\n")
    else:
        print("No valid multi-city flights found under the price limit.")
except FileNotFoundError:
    print("No multi_city_results.json file found. Run parse_multi_city.py first.")



✈️  Best Multi-City Flights found under 75000 rub:

Option 1:
   -  flight: MOW-TYO + TYO-MOW
   - 💰 Price: 57672 rub
   - ⏳ Total Transfers (both ways): 2
   - 🛫 Departure Date: 2026-07-31
   - 🛬 Return Date: 2026-08-10
   - 🔗 Ticket Link: https://avs.io/m2Xq

Option 2:
   -  flight: MOW-TYO + TYO-MOW
   - 💰 Price: 57672 rub
   - ⏳ Total Transfers (both ways): 2
   - 🛫 Departure Date: 2026-07-30
   - 🛬 Return Date: 2026-08-10
   - 🔗 Ticket Link: https://avs.io/m2YW

Option 3:
   -  flight: MOW-TYO + TYO-MOW
   - 💰 Price: 57672 rub
   - ⏳ Total Transfers (both ways): 2
   - 🛫 Departure Date: 2026-07-31
   - 🛬 Return Date: 2026-08-11
   - 🔗 Ticket Link: https://avs.io/m2Z9

Option 4:
   -  flight: MOW-TYO + TYO-MOW
   - 💰 Price: 57672 rub
   - ⏳ Total Transfers (both ways): 2
   - 🛫 Departure Date: 2026-07-29
   - 🛬 Return Date: 2026-08-10
   - 🔗 Ticket Link: https://avs.io/m31A

Option 5:
   -  flight: MOW-TYO + TYO-MOW
   - 💰 Price: 57672 rub
   - ⏳ Total Transfers (both ways): 2
   